<a href="https://colab.research.google.com/github/susmitsingh01/cuda-triton-llm-kernels/blob/main/01_vadd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 : Vector Add

## Step 0: Prediction
- Operation: c = a + b, n elements, fp32
- Bytes moved: read a, b (8 bytes) + write c (4 bytes) = **12n bytes**
- Work: **n FLOPs** (one add per element)
- Arithmetic intensity: 1/12 ≈ **0.083 FLOP/byte**
  (T4 balance point ≈ 8 TFLOP/s ÷ 320 GB/s ≈ 25 FLOP/byte)
- **Prediction: memory-bound.** Best possible time ≈ 12n ÷ achievable bandwidth.
- Target: effective bandwidth as % of measured peak (copy kernel, measured later).

## Environment logging




In [1]:
import platform
import subprocess

import torch
import triton


def env_info():
    smi = subprocess.run(
        ["nvidia-smi",
         "--query-gpu=name,driver_version,clocks.sm,clocks.mem,ecc.mode.current",
         "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip()
    return {
        "gpu": smi,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "triton": triton.__version__,
        "python": platform.python_version(),
    }

In [4]:
env_info()

{'gpu': 'Tesla T4, 580.82.07, 300 MHz, 405 MHz, Enabled',
 'torch': '2.11.0+cu128',
 'cuda': '12.8',
 'triton': '3.6.0',
 'python': '3.13.15'}

## PyTorch reference

In [5]:
import torch

def vadd_ref(a, b):
    return a + b

# sanity check
a = torch.randn(1024, device="cuda")
b = torch.randn(1024, device="cuda")
print(torch.equal(vadd_ref(a, b), a + b))

True
